# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook demonstrates how to load, process, and analyze the FAIR² dataset using the `mlcroissant` library.

**Dataset title**: Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution

### Dataset Source
The dataset is described using the [Croissant schema](https://mlcommons.org/croissant/) and is accessible by its schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Date Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Explore what record sets and fields are present in the dataset. All entities, such as record sets and fields, are referenced by their `@id` fields.

In [ ]:
# List all available record sets (`@id`s and names)

print('Available record sets:')
for rs in dataset.record_sets:
    print(f"  @id: {rs.id} | name: {getattr(rs, 'name', 'N/A')}")

# Explore fields for each record set

for rs in dataset.record_sets:
    print(f"\nRecord Set @id: {rs.id} (name: {getattr(rs, 'name', 'N/A')})")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields (by @id):")
        for f in rs.fields:
            print(f"    - {f.id} | name: {getattr(f, 'name', 'N/A')} | type: {getattr(f, 'data_type', 'N/A')}")
    else:
        print("  No fields found.")

## 3. Data Extraction
Load each record set (using `@id`) into pandas DataFrames for further analysis. You should use the exact record set and field `@id`s as listed above.

In [ ]:
# Prepare to extract all available record sets as DataFrames

# Gather all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading record set {rs_id} ...")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        if not df.empty:
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("  No records found.")
        dataframes[rs_id] = df
    except Exception as e:
        print(f"  Failed to load: {e}")

# Show the columns in the first available record set
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nFirst loaded DataFrame columns ({first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No non-empty DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply basic analysis: filter records, normalize numeric fields, group by categorical variables. Use only field and record set `@id`s for references.

In [ ]:
# Choose the main record set for EDA. If known, specify, or default to first one loaded.

if dataframes:
    # Identify a numeric field automatically (fallback if unknown)
    rs_id = first_rs_id
    df = dataframes[rs_id]

    # Try to select a numeric field (by dtype)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
    else:
        print('No numeric fields found; unable to perform numeric EDA.')
        numeric_field = None
    
    if numeric_field is not None:
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f}: {len(filtered_df)} records")

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"First normalized values for '{numeric_field}':")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to pick a categorical/grouping field
        candidate_group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        group_field = None
        for col in candidate_group_fields:
            nunique = df[col].nunique()
            if nunique > 1 and nunique <= 10:
                group_field = col
                break

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Mean {numeric_field} grouped by '{group_field}':")
            print(grouped_df.head())
        else:
            print('No suitable group field found for grouping.')
    else:
        print('Skipping EDA numeric and group analysis.')
else:
    print('No loaded DataFrames to perform EDA.')

## 5. Visualization
Visualize data distributions and relationships using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field's distribution, if available
if dataframes and numeric_field is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
    
    # If group_field is available, plot boxplot/group differences
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² colorectal cancer survivors dataset via its Croissant schema, explored available record sets and fields by their `@id`s, and performed automatic basic exploratory analysis and data visualization. The notebook provides a template for deeper analysis and reproducible workflows on Croissant-based biomedical datasets.